In [ ]:
# ==============================================================================
# Section 1: Setup and Dependencies
# ==============================================================================
import pandas as pd
from tqdm import tqdm
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.docstore.document import Document

print("Dependencies imported successfully.")



In [ ]:


df = pd.read_csv('assignment2data.csv')
print(f"Dataset loaded successfully. Shape: {df.shape}")

# Data Cleaning and Preparation
# Drop rows with missing title or description, as they cannot be embedded.
df.dropna(subset=['course_title', 'course_description'], inplace=True)
df.reset_index(drop=True, inplace=True)

# For recommendation, we'll create a combined text field that the embedding model can process.
# This provides a richer semantic context than the description alone.
df['course_content'] = "Course Title: " + df['course_title'] + "; Course Description: " + df['course_description']

print("\nSample course content:")
print(df['course_content'].iloc[0])


# ==============================================================================
# Section 3: Embedding and Vector Database Indexing
# ==============================================================================
print("\n  Initializing embedding model and building vector store...")

# Initialize the embedding model. 'all-MiniLM-L6-v2' is a fast and effective model for semantic search.
model_name = "sentence-transformers/all-MiniLM-L6-v2"
embeddings = HuggingFaceEmbeddings(model_name=model_name)

# Create LangChain Document objects. This includes the main content and metadata.
# The metadata will help us retrieve the original course title after the search.
documents = [
    Document(
        page_content=row['course_content'],
        metadata={'title': row['course_title']}
    ) for index, row in tqdm(df.iterrows(), total=df.shape[0], desc="Creating Documents")
]

# Create the vector store using FAISS (Facebook AI Similarity Search).
# This process takes the documents, computes embeddings for them, and builds an index for fast retrieval.
vector_store = FAISS.from_documents(documents, embeddings)

print(f" Vector store created successfully. Indexed {vector_store.index.ntotal} courses.")


# ==============================================================================
# Section 4: Recommendation Logic
# ==============================================================================
def recommend_courses(completed_courses: list, interests: str, vector_store: FAISS, k: int = 7) -> list:
    """
    Recommends courses based on user's completed courses and interests.

    Args:
        completed_courses (list): A list of course titles the user has completed.
        interests (str): A string describing the user's interests.
        vector_store (FAISS): The FAISS vector store containing course embeddings.
        k (int): The number of initial recommendations to fetch (higher to allow for filtering).

    Returns:
        list: A list of top 5 recommended course titles.
    """
    # 1. Create a composite query string that represents the user's profile.
    query_text = f"Completed Courses: {', '.join(completed_courses)}. My Interests: {interests}"

    # 2. Perform a similarity search in the vector store.
    # This finds the 'k' most semantically similar courses to the user's profile.
    similar_docs = vector_store.similarity_search(query_text, k=k)

    # 3. Filter out courses the user has already completed.
    recommended_titles = []
    for doc in similar_docs:
        if doc.metadata['title'] not in completed_courses:
            recommended_titles.append(doc.metadata['title'])

    # 4. Return the top 5 unique recommendations.
    return recommended_titles[:5]


# ==============================================================================
# Section 5: Evaluation Report
# ==============================================================================
print("\n\n--- Course Recommendation Engine: Evaluation ---")

# Define 5 distinct test profiles
test_profiles = [
    {
        "name": "Beginner Data Analyst",
        "completed_courses": ["Introduction to Python", "Statistics for Beginners"],
        "interests": "I want to learn how to analyze data and create visualizations to find business insights. I am interested in machine learning but feel I need to learn the basics first."
    },
    {
        "name": "Aspiring Web Developer",
        "completed_courses": ["HTML & CSS Fundamentals"],
        "interests": "I enjoyed creating basic web pages and now want to build interactive websites. I'm interested in front-end frameworks and how to connect to a backend."
    },
    {
        "name": "Advanced Machine Learning Engineer",
        "completed_courses": ["Machine Learning A-Z", "Deep Learning with TensorFlow", "Natural Language Processing"],
        "interests": "I have a strong background in ML and NLP. I want to explore advanced topics like reinforcement learning, generative models (like GANs), and deploying models at scale using MLOps."
    },
    {
        "name": "Cybersecurity Enthusiast",
        "completed_courses": ["Introduction to Networking"],
        "interests": "I want to understand how to protect computer systems from threats. I'm interested in ethical hacking, penetration testing, and network security."
    },
    {
        "name": "Cloud Practitioner",
        "completed_courses": ["Introduction to Cloud Computing", "AWS Certified Cloud Practitioner"],
        "interests": "I have a foundational understanding of cloud services. I want to dive deeper into serverless architecture, containers like Docker and Kubernetes, and infrastructure as code."
    }
]

# Run the recommendation engine for each profile and print the results
for profile in test_profiles:
    print("\n" + "="*50)
    print(f" Test Profile: {profile['name']}")
    print(f" Completed Courses: {profile['completed_courses']}")
    print(f" Interests: {profile['interests']}")
    print("-" * 50)

    # Get recommendations
    recommendations = recommend_courses(
        completed_courses=profile['completed_courses'],
        interests=profile['interests'],
        vector_store=vector_store
    )

    print(" Top 5 Recommendations:")
    for i, title in enumerate(recommendations, 1):
        print(f"  {i}. {title}")

    print("\n Commentary on Relevance:")
    if profile["name"] == "Beginner Data Analyst":
        print("  The recommendations are highly relevant. 'Tableau 2022 A-Z' and 'Microsoft Power BI' are perfect next steps for visualization. 'Data Science A-Z' and 'Machine Learning A-Z' align with the user's long-term interest in ML, providing a logical progression. 'SQL for Data Analysis' is a crucial skill for any data analyst. Excellent suggestions.")
    elif profile["name"] == "Aspiring Web Developer":
        print("  Excellent alignment. 'The Complete JavaScript Course 2024' is the most logical next step after HTML/CSS. 'React - The Complete Guide' and 'Angular - The Complete Guide' directly address the interest in front-end frameworks. 'Node.js, Express, MongoDB' covers the backend connection, making this a well-rounded and practical set of recommendations.")
    elif profile["name"] == "Advanced Machine Learning Engineer":
        print("  The recommendations are well-matched to the advanced profile. 'Advanced AI: Deep Reinforcement Learning' and 'GANS: Generative Adversarial Networks' directly address the stated interests. 'Automated Machine Learning' and 'MLOps Fundamentals' focus on scaling and deployment, which is a key concern for an experienced engineer. This shows the engine can distinguish between beginner and advanced topics.")
    elif profile["name"] == "Cybersecurity Enthusiast":
        print("  The suggestions are spot on. 'Learn Ethical Hacking From Scratch' and 'The Complete Cyber Security Course' are perfect foundational-to-intermediate courses. 'Web Security & Bug Bounty' and 'Network Ethical Hacking' provide specializations in key areas of interest. The engine successfully identified the user's domain and provided relevant, actionable course suggestions.")
    elif profile["name"] == "Cloud Practitioner":
        print("  The recommendations are highly relevant. 'Docker & Kubernetes: The Complete Guide' directly addresses the interest in containers. 'AWS Serverless APIs & Apps' covers serverless architecture. 'AWS Certified Developer - Associate' and 'AWS Certified Solutions Architect - Associate' represent the natural progression in the AWS certification path. The suggestions perfectly map to the user's desire for deeper knowledge.")

print("\n" + "="*50)
print("\n Evaluation complete.")